# Gold Layer - Business Metrics

Aggregates silver data into analytical dimensions for reporting.

**Source:** workspace.silver_chocolate  
**Target:** workspace.gold_chocolate

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, upper, lower, when, coalesce,
    current_timestamp, lit, count, countDistinct, sum as spark_sum, avg, max as spark_max, min as spark_min,
    round as spark_round, dense_rank, row_number, concat_ws, md5
)
from pyspark.sql.window import Window
from pyspark.sql.types import *

# Configuration
SILVER_CATALOG = "workspace"
SILVER_SCHEMA = "silver_chocolate"
GOLD_CATALOG = "workspace"
GOLD_SCHEMA = "gold_chocolate"



✓ Imports loaded
✓ Source: workspace.silver_chocolate
✓ Target: workspace.gold_chocolate


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}")

✓ Schema workspace.gold_chocolate ready


In [0]:


df_sales = spark.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_sales")
df_products = spark.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_products")
df_stores = spark.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_stores")
df_geography = spark.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_geography")




📊 Loading Silver tables...
  • Sales: 1,000,000 rows
  • Products: 202 rows
  • Stores: 100 rows
  • Geography: 6 rows


In [0]:



df_fact_sales = df_sales.alias("s") \
    .join(df_geography.alias("g"), df_sales.country == df_geography.country, "left")


df_fact_sales = df_fact_sales.withColumn(
    "sale_id",
    row_number().over(Window.orderBy("sale_date", "order_id"))
)


df_gold_fact_sales = df_fact_sales.select(

    col("sale_id"),
    col("s.order_id").alias("order_id"),
    col("s.product_id").alias("product_id"),
    col("s.store_id").alias("store_id"),
    col("s.customer_id").alias("customer_id"),
    col("g.country_id").alias("country_id"),
    

    col("s.sale_date").alias("sale_date"),
    col("s.year").alias("year"),
    col("s.quarter").alias("quarter"),
    col("s.month").alias("month"),
    col("s.day").alias("day"),
    

    col("s.product_name").alias("product_name"),
    col("s.brand").alias("brand"),
    col("s.category").alias("category"),
    

    col("s.store_name").alias("store_name"),
    col("s.city").alias("city"),
    col("s.country").alias("country"),
    col("s.store_type").alias("store_type"),
    

    col("s.age").alias("age"),
    col("s.gender").alias("gender"),
    col("s.loyalty_member").alias("loyalty_member"),
    

    col("s.quantity_clean").alias("quantity"),
    col("s.unit_price_clean").alias("unit_price"),
    col("s.discount_clean").alias("discount"),
    col("s.discount_amount").alias("discount_amount"),
    col("s.revenue_clean").alias("revenue"),
    col("s.cost_clean").alias("cost"),
    col("s.profit_clean").alias("profit"),
    spark_round(col("s.profit_margin_pct"), 2).alias("profit_margin_pct"),
    

    col("s.is_valid").alias("is_valid"),
    current_timestamp().alias("gold_processed_timestamp")
)




gold_fact_sales_path = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_fact_sales"
df_gold_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_fact_sales_path)

print(f"Gold fact sales: {df_gold_fact_sales.count():,} rows -> {gold_fact_sales_path}")


🎯 Creating Gold Fact Sales (denormalized)...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  • Fact Sales rows: 1,000,000
  • Columns: 31
  ✓ Saved to: workspace.gold_chocolate.gold_fact_sales


In [0]:


from pyspark.sql.functions import dayofweek, weekofyear, date_format

df_date_dim = df_sales.select("sale_date", "year", "quarter", "month", "day").distinct() \
    .withColumn("date_id", row_number().over(Window.orderBy("sale_date"))) \
    .withColumn("day_of_week", dayofweek(col("sale_date"))) \
    .withColumn("week_of_year", weekofyear(col("sale_date"))) \
    .withColumn("day_name", date_format(col("sale_date"), "EEEE")) \
    .withColumn("month_name", date_format(col("sale_date"), "MMMM")) \
    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), True).otherwise(False)) \
    .withColumn("gold_processed_timestamp", current_timestamp())

df_gold_date = df_date_dim.select(
    "date_id", "sale_date", "year", "quarter", "month", "day",
    "day_of_week", "day_name", "week_of_year", "month_name", "is_weekend",
    "gold_processed_timestamp"
).orderBy("sale_date")



gold_dim_date_path = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_dim_date"
df_gold_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_dim_date_path)

print(f"Gold date dimension: {df_gold_date.count():,} rows -> {gold_dim_date_path}")


📅 Creating Gold Date Dimension...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  • Date dimension rows: 731
  ✓ Saved to: workspace.gold_chocolate.gold_dim_date


In [0]:



product_metrics = df_sales \
    .groupBy("product_id") \
    .agg(
        spark_sum("quantity_clean").alias("total_quantity_sold"),
        spark_sum("revenue_clean").alias("total_revenue"),
        spark_sum("profit_clean").alias("total_profit"),
        avg("profit_margin_pct").alias("avg_profit_margin"),
        count("*").alias("total_transactions")
    )

df_gold_product = df_products \
    .join(product_metrics, "product_id", "left") \
    .withColumn("gold_processed_timestamp", current_timestamp()) \
    .select(
        "product_id",
        "product_name",
        "product_name_clean",
        "brand",
        "brand_clean",
        "category",
        "category_clean",
        coalesce("total_quantity_sold", lit(0)).alias("total_quantity_sold"),
        coalesce(spark_round("total_revenue", 2), lit(0.0)).alias("total_revenue"),
        coalesce(spark_round("total_profit", 2), lit(0.0)).alias("total_profit"),
        coalesce(spark_round("avg_profit_margin", 2), lit(0.0)).alias("avg_profit_margin_pct"),
        coalesce("total_transactions", lit(0)).alias("total_transactions"),
        "gold_processed_timestamp"
    )



gold_dim_product_path = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_dim_product"
df_gold_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_dim_product_path)

print(f"Gold product dimension: {df_gold_product.count():,} rows -> {gold_dim_product_path}")


🍫 Creating Gold Product Dimension...
  • Product dimension rows: 202
  ✓ Saved to: workspace.gold_chocolate.gold_dim_product


In [0]:



store_metrics = df_sales \
    .groupBy("store_id") \
    .agg(
        spark_sum("quantity_clean").alias("total_quantity_sold"),
        spark_sum("revenue_clean").alias("total_revenue"),
        spark_sum("profit_clean").alias("total_profit"),
        avg("profit_margin_pct").alias("avg_profit_margin"),
        count("*").alias("total_sales_count"),
        countDistinct("customer_id").alias("unique_customers")
    )

df_gold_store = df_stores \
    .join(store_metrics, "store_id", "left") \
    .withColumn("gold_processed_timestamp", current_timestamp()) \
    .select(
        "store_id",
        "store_name",
        "store_name_clean",
        "city",
        "city_clean",
        "country",
        "country_clean",
        "store_type",
        "store_type_clean",
        coalesce("total_quantity_sold", lit(0)).alias("total_quantity_sold"),
        coalesce(spark_round("total_revenue", 2), lit(0.0)).alias("total_revenue"),
        coalesce(spark_round("total_profit", 2), lit(0.0)).alias("total_profit"),
        coalesce(spark_round("avg_profit_margin", 2), lit(0.0)).alias("avg_profit_margin_pct"),
        coalesce("total_sales_count", lit(0)).alias("total_sales_count"),
        coalesce("unique_customers", lit(0)).alias("unique_customers"),
        "gold_processed_timestamp"
    )



gold_dim_store_path = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_dim_store"
df_gold_store.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_dim_store_path)

print(f"Gold store dimension: {df_gold_store.count():,} rows -> {gold_dim_store_path}")


🏪 Creating Gold Store Dimension...
  • Store dimension rows: 100
  ✓ Saved to: workspace.gold_chocolate.gold_dim_store


In [0]:



geography_metrics = df_sales \
    .groupBy("country") \
    .agg(
        spark_sum("quantity_clean").alias("total_quantity_sold"),
        spark_sum("revenue_clean").alias("total_revenue"),
        spark_sum("profit_clean").alias("total_profit"),
        avg("profit_margin_pct").alias("avg_profit_margin"),
        count("*").alias("total_transactions"),
        countDistinct("store_id").alias("store_count"),
        countDistinct("customer_id").alias("customer_count")
    )

df_gold_geography = df_geography \
    .join(geography_metrics, "country", "left") \
    .withColumn("gold_processed_timestamp", current_timestamp()) \
    .select(
        "country_id",
        "country",
        "country_name",
        coalesce("total_quantity_sold", lit(0)).alias("total_quantity_sold"),
        coalesce(spark_round("total_revenue", 2), lit(0.0)).alias("total_revenue"),
        coalesce(spark_round("total_profit", 2), lit(0.0)).alias("total_profit"),
        coalesce(spark_round("avg_profit_margin", 2), lit(0.0)).alias("avg_profit_margin_pct"),
        coalesce("total_transactions", lit(0)).alias("total_transactions"),
        coalesce("store_count", lit(0)).alias("store_count"),
        coalesce("customer_count", lit(0)).alias("customer_count"),
        "gold_processed_timestamp"
    )



gold_dim_geography_path = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_dim_geography"
df_gold_geography.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_dim_geography_path)

print(f"Gold geography dimension: {df_gold_geography.count():,} rows -> {gold_dim_geography_path}")


🌍 Creating Gold Geography Dimension...
  • Geography dimension rows: 6
  ✓ Saved to: workspace.gold_chocolate.gold_dim_geography


In [0]:
print("\nGold layer summary:")

gold_tables = [
    "gold_fact_sales",
    "gold_dim_date",
    "gold_dim_product",
    "gold_dim_store",
    "gold_dim_geography"
]

for table_name in gold_tables:
    table_path = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.{table_name}"
    df = spark.table(table_path)
    row_count = df.count()
    col_count = len(df.columns)
    print(f"{table_path}: {row_count:,} rows, {col_count} columns")




GOLD LAYER TABLES
  • workspace.gold_chocolate.gold_fact_sales: 1,000,000 rows, 31 columns
  • workspace.gold_chocolate.gold_dim_date: 731 rows, 12 columns
  • workspace.gold_chocolate.gold_dim_product: 202 rows, 13 columns
  • workspace.gold_chocolate.gold_dim_store: 100 rows, 16 columns
  • workspace.gold_chocolate.gold_dim_geography: 6 rows, 11 columns

✓ Gold tables are ready for Star Schema consumption!
